In [21]:
from torch import nn
import pandas as pd
from torch.utils.data import Dataset
from sklearn.preprocessing import StandardScaler
from train_utils import train_model

In [22]:
class LogisticRegressor(nn.Module):
    def __init__(self, in_features):
        """
        Input:
        in_features: numero di feature in input (es. 13 per il nostro dataset)
        """
        # Richiamiamo il costruttore della superclasse
        super(LogisticRegressor, self).__init__()
        
        # Definiamo la trasformazione lineare: y = Ax + b
        self.linear = nn.Linear(in_features, 1)
        
    def forward(self, x):
        # Calcoliamo e restituiamo i logit
        logits = self.linear(x)
        return logits



In [23]:
!pip install kagglehub

In [24]:
# Cella 1: Setup e Autoreload
%load_ext autoreload
%autoreload 2

import torch
from torch.utils.data import DataLoader, random_split

# Importiamo la nostra classe custom dal file dataset.py
from dataset import HeartDiseaseDataset

# Definiamo i parametri e carichiamo i dati
csv_path = "data/heart.csv"  # Assicurati che il path sia corretto
dataset = HeartDiseaseDataset(csv_path)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Dati caricati! Feature in ingresso: {dataset[0][0].shape[0]}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Dati caricati! Feature in ingresso: 13


In [25]:
model_name = 'linear_regressor'

# Ricaviamo dinamicamente il numero di feature (es. 13) leggendo il primo campione
input_dim = dataset[0][0].shape[0] 

# 3. Prepariamo lo split e i DataLoader (come definito nel blocco precedente)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 4. Istanziamo il modello base
print(f"Inizializzazione del LogisticRegressor con {input_dim} feature in ingresso...")
model = LogisticRegressor(in_features=input_dim)

# 5. Facciamo partire l'addestramento
print("Inizio dell'addestramento...")
trained_model = train_model(
    model=model, 
    train_loader=train_loader, 
    test_loader=test_loader, 
    name_model=model_name,
    epochs=100, 
    lr=0.01
)
print("Addestramento completato con successo!")

Inizializzazione del LogisticRegressor con 13 feature in ingresso...
Inizio dell'addestramento...
Addestramento completato con successo!


In [26]:
from metrics_report import evaluate_and_save


evaluate_and_save(trained_model, test_loader, model_name=model_name)


--- Report: linear_regressor ---
Accuracy:  0.8585
Precision: 0.8250
Recall:    0.9252
F1-Score:  0.8722
Matrice di Confusione:
[[77 21]
 [ 8 99]]

Risultati di linear_regressor salvati in results.csv
